# Lyrics Enrichment and Feature Engineering

This notebook enriches the rock song dataset by incorporating lyrical data using the Genius API.

The goal of this stage is to:
- retrieve song lyrics for a subset of the dataset,
- clean and preprocess the text data,
- extract meaningful numerical features from lyrics,
- and prepare the dataset for machine learning analysis.

Since the original dataset contains only audio features, this enrichment step allows us to investigate whether lyrical content contributes to predicting hit songs.

## Loading Dataset and Initial Setup

The cleaned rock dataset is loaded, and a Genius API client is initialized to retrieve song lyrics.

Due to API limitations and efficiency considerations, only a subset of the dataset will be used for enrichment.

In [1]:
import pandas as pd
import lyricsgenius
from tqdm import tqdm

# your token
genius = lyricsgenius.Genius("JqrylRrVHLexjcbQNSD26zgRdVKZB_eqNH44MaJeIyxVquzIwkDbZV13hJmJ_WGv")

rock_df = pd.read_csv("../data/clean/rock_dataset_cleaned.csv")
rock_df.head()

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,...,instrumentalness,liveness,valence,tempo,time_signature,track_genre,is_hit,track_name_length,artist_count,duration_min
0,4h9wh7iOZ0GGn8QVp4RAOB,OneRepublic,I Ain’t Worried (Music From The Motion Picture...,I Ain't Worried,96,148485,False,0.704,0.797,0,...,0.000745,0.0546,0.825,139.994,4,rock,1,15,1,2.474750
1,2QjOHCTQ1Jl3zawyYOpxh6,The Neighbourhood,I Love You.,Sweater Weather,93,240400,False,0.612,0.807,10,...,0.017700,0.1010,0.398,124.053,4,rock,1,15,1,4.006667
2,5XeFesFbtLpXzIVDNQP22n,Arctic Monkeys,AM,I Wanna Be Yours,92,183956,False,0.464,0.417,0,...,0.022000,0.0974,0.479,67.528,4,rock,1,16,1,3.065933
3,75FEaRjZTKLhTrFGsfMUXR,Kate Bush,Hounds Of Love,Running Up That Hill (A Deal With God),90,298933,False,0.629,0.547,10,...,0.003140,0.0604,0.197,108.375,4,rock,1,38,1,4.982217
4,7DbdUf8aHSYoliSjO6LZv6,Beach Weather,Chit Chat,"Sex, Drugs, Etc.",90,196784,False,0.572,0.839,4,...,0.009760,0.5220,0.465,143.969,4,rock,1,16,1,3.279733


## Sampling the Dataset

A random sample of 300 songs is selected from the full dataset.

This is done to:
- reduce API usage,
- avoid rate limiting,
- and allow faster experimentation during development.

In [2]:
sample_df = rock_df.sample(300, random_state=42).copy()

## Retrieving Lyrics from Genius API

Lyrics are retrieved using the Genius API by matching each song's title and artist.

Due to inconsistencies in metadata and API limitations:
- some songs may not return lyrics,
- some matches may not be perfect.

Errors are handled gracefully to ensure the process continues even if some requests fail.

In [3]:
def get_lyrics(row):
    try:
        song = genius.search_song(row["track_name"])
        if song:
            return song.lyrics
    except:
        return None
    return None

lyrics_list = []

for _, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
    lyrics = get_lyrics(row)
    lyrics_list.append(lyrics)

sample_df["lyrics"] = lyrics_list

100%|██████████| 300/300 [14:52<00:00,  2.98s/it]


## Saving Enriched Dataset

The dataset with retrieved lyrics is saved for reuse, avoiding repeated API calls in future runs.

In [4]:
sample_df.to_csv("../data/enriched/rock_with_lyrics_sample.csv", index=False)

## Evaluating Lyrics Retrieval Success

The proportion of missing lyrics is calculated to assess the effectiveness of the enrichment process.

A small percentage of missing values is expected due to unmatched songs or API limitations.

In [5]:
sample_df["lyrics"].isna().mean()

np.float64(0.04666666666666667)

## Initial Lyrics Length Calculation

A basic feature representing the number of words in each song is computed.

This serves as a simple measure of lyrical complexity or verbosity.

In [6]:
sample_df["lyrics_length"] = sample_df["lyrics"].apply(
    lambda x: len(x.split()) if isinstance(x, str) else 0
)

## Manual Inspection of Retrieved Lyrics

A sample of songs and their corresponding lyrics is displayed to verify that:
- the correct songs are being matched,
- the lyrics are properly retrieved,
- and no obvious mismatches exist.

In [7]:
sample_df[["track_name", "artists", "lyrics"]].head(20)

,track_name,artists,lyrics
5368,Runaway Horses - Abridged,The Killers;Phoebe Bridgers,Lizzo - Rumors (feat. Cardi B)\nPinkPantheress...
4328,Mis Noches Sin Ti,Los Fredy´s,Mis Noches Sin Ti\nSufro al pensar que el dest...
199,Stayin' Alive,Bee Gees,"[Verse 1]\nWell, you can tell by the way I use..."
3264,Popotitos (Bonie Moroni),Los Teen Tops,Mi amor entero es de mi novia popotitos\nSus p...
4522,Ten Commandments Of Love,The Moonglows,[Verse 1]\n(One)\nThou shalt never love anothe...
2552,O Último Bar,Matanza,O último bar quando fecha de manhã\nSó me lemb...
2767,今今ここに君とあたし,Creep Hyp,「昔昔あるところに」って曖昧過ぎてわからなくて\nおじいさんとおばあさんじゃ何だか物足りなく...
2209,Fusilados por la Cruz Roja,Patricio Rey y sus Redonditos de Ricota,"[Letra de ""Fusilados por la Cruz Roja""]\n\n[In..."
1212,Wicked Ones,Dorothy,[Verse 1]\nThis night ain't for the faint of h...
5593,Echo,Tom Petty and the Heartbreakers,"[Intro]\nDamn, Callan\n (\nDamn, Callan\n)\n\n..."


## Cleaning Lyrics Text

The raw lyrics are cleaned to prepare them for analysis:
- text is converted to lowercase,
- section markers such as [Chorus] and [Verse] are removed,
- non-alphabetic characters are removed.

This ensures consistency and improves the quality of feature extraction.

In [8]:
import re

def clean_lyrics(text):
    if not isinstance(text, str):
        return ""
    
    text = text.lower()
    text = re.sub(r"\[.*?\]", "", text)  # remove [Chorus], [Verse]
    text = re.sub(r"[^a-z\s]", "", text)  # keep only letters
    text = text.replace("\n", " ")
    return text

sample_df["clean_lyrics"] = sample_df["lyrics"].apply(clean_lyrics)

## Recomputing Lyrics Length

Lyrics length is recalculated after cleaning to ensure accurate word counts based on processed text.

In [9]:
sample_df["lyrics_length"] = sample_df["clean_lyrics"].apply(lambda x: len(x.split()))

## Unique Word Count

The number of unique words in each song is calculated.

This provides a measure of vocabulary richness in the lyrics.

In [10]:
sample_df["unique_words"] = sample_df["clean_lyrics"].apply(
    lambda x: len(set(x.split()))
)

## Lexical Diversity

Lexical diversity is computed as the ratio of unique words to total words.

This captures how varied the vocabulary is within a song:
- higher values indicate more diverse language,
- lower values indicate repetition.

In [11]:
sample_df["lexical_diversity"] = (
    sample_df["unique_words"] / sample_df["lyrics_length"].replace(0, 1)
)

## Sentiment Analysis

Sentiment scores are computed using TextBlob.

Each song is assigned a polarity score:
- negative values indicate negative sentiment,
- positive values indicate positive sentiment.

This feature represents the emotional tone of the lyrics.

In [12]:
from textblob import TextBlob

def get_sentiment(text):
    if text == "":
        return 0
    return TextBlob(text).sentiment.polarity

sample_df["sentiment"] = sample_df["clean_lyrics"].apply(get_sentiment)

## Saving Final Enriched Dataset

The dataset with extracted lyrical features is saved for use in machine learning models.

In [13]:
sample_df.to_csv("../data/enriched/rock_with_lyrics_features.csv", index=False)

## Summary Statistics of Lyrical Features

Summary statistics are examined to understand the distribution of:
- lyrics length,
- lexical diversity,
- sentiment scores.

This helps identify potential outliers and data quality issues.

In [14]:
sample_df[["lyrics_length", "lexical_diversity", "sentiment"]].describe()

,lyrics_length,lexical_diversity,sentiment
count,300.0000,300.000000,300.000000
mean,415.0000,0.375629,0.038151
std,1560.9082,0.181326,0.155697
min,0.0000,0.000000,-0.517857
25%,124.0000,0.265401,0.000000
50%,211.0000,0.377218,0.000000
75%,328.5000,0.478231,0.110158
max,21745.0000,1.000000,0.600000


## Removing Extreme Outliers

Songs with extremely large lyrics length values are removed, as these may result from:
- incorrect scraping,
- repeated or duplicated text,
- or non-standard entries.

This improves the robustness of downstream analysis.

In [15]:
sample_df = sample_df[sample_df["lyrics_length"] < 5000]

## Removing Very Short Lyrics

Songs with extremely short lyrics are removed, as they may:
- distort lexical diversity calculations,
- or represent incomplete or missing data.

In [16]:
sample_df = sample_df[sample_df["lyrics_length"] > 20]

In [17]:
sample_df["lyrics_length"].describe()

count     267.000000
mean      282.674157
std       243.142496
min        27.000000
25%       147.000000
50%       228.000000
75%       334.500000
max      2218.000000
Name: lyrics_length, dtype: float64

## Capping Extreme Values

Lyrics length is capped at an upper threshold to reduce the impact of remaining outliers.

This prevents extreme values from disproportionately influencing machine learning models.

In [18]:
sample_df["lyrics_length"] = sample_df["lyrics_length"].clip(upper=2000)

In [19]:
sample_df["lyrics_length"].describe()

count     267.000000
mean      281.857678
std       236.905208
min        27.000000
25%       147.000000
50%       228.000000
75%       334.500000
max      2000.000000
Name: lyrics_length, dtype: float64

In [20]:
sample_df[["track_name", "artists", "clean_lyrics"]].head(20)

,track_name,artists,clean_lyrics
5368,Runaway Horses - Abridged,The Killers;Phoebe Bridgers,lizzo rumors feat cardi b pinkpantheress jus...
4328,Mis Noches Sin Ti,Los Fredy´s,mis noches sin ti sufro al pensar que el desti...
199,Stayin' Alive,Bee Gees,well you can tell by the way i use my walk im...
3264,Popotitos (Bonie Moroni),Los Teen Tops,mi amor entero es de mi novia popotitos sus pi...
4522,Ten Commandments Of Love,The Moonglows,one thou shalt never love another two but sta...
2552,O Último Bar,Matanza,o ltimo bar quando fecha de manh s me lembra q...
2209,Fusilados por la Cruz Roja,Patricio Rey y sus Redonditos de Ricota,checheche checheche hay hay mucho misteri...
1212,Wicked Ones,Dorothy,this night aint for the faint of heart for th...
5593,Echo,Tom Petty and the Heartbreakers,damn callan damn callan one bad bitch an...
1397,Good Morning World!,BURNOUT SYNDROMES,good morning world good morning world ...


## Conclusion

The dataset has been successfully enriched with lyrical features derived from song lyrics.

Key features extracted include:
- lyrics length,
- lexical diversity,
- and sentiment.

These features will be combined with audio features in the next stage to evaluate whether lyrical content improves the prediction of hit songs.